# FER Emotion Recognition — Kaggle (Folder-Based)
### Model 1: Custom CNN (trained from scratch)
### Model 2: MobileNetV2 Transfer Learning (fully trainable + aggressive LR decay)

**Dataset:** `/kaggle/input/fer2013/train` & `/kaggle/input/fer2013/test`  
**Classes:** Angry, Happy, Sad, Surprise, Neutral (5 classes)  
**Input size:** 128×128 RGB

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, regularizers
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Input, Conv2D, BatchNormalization, MaxPooling2D,
    GlobalAveragePooling2D, Dense, Dropout, Flatten
)
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## Configuration

In [ ]:
TRAIN_DIR  = '/kaggle/input/fer2013/train'
TEST_DIR   = '/kaggle/input/fer2013/test'

IMG_SIZE   = (128, 128)
BATCH_SIZE = 32
EPOCHS     = 40
NUM_CLASSES = 5

EMOTIONS = ['angry', 'happy', 'sad', 'surprise', 'neutral']

---
# Part 1 — Custom CNN Baseline
## 1.1 Data Generators for CNN

In [ ]:
# CNN uses simple [0,1] normalisation
cnn_train_gen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

cnn_train_data = cnn_train_gen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    classes=EMOTIONS
)

cnn_val_data = cnn_train_gen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    classes=EMOTIONS
)

cnn_test_gen = ImageDataGenerator(rescale=1./255)
cnn_test_data = cnn_test_gen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
    classes=EMOTIONS
)

## 1.2 Build CNN Model

In [ ]:
def build_cnn(input_shape=(128, 128, 3), num_classes=5):
    l2 = regularizers.L2(1e-4)
    inp = Input(shape=input_shape)

    # Block 1
    x = Conv2D(32, (3,3), padding='same', activation='relu', kernel_regularizer=l2)(inp)
    x = BatchNormalization()(x)
    x = Conv2D(32, (3,3), padding='same', activation='relu', kernel_regularizer=l2)(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D(2,2)(x)
    x = Dropout(0.25)(x)

    # Block 2
    x = Conv2D(64, (3,3), padding='same', activation='relu', kernel_regularizer=l2)(x)
    x = BatchNormalization()(x)
    x = Conv2D(64, (3,3), padding='same', activation='relu', kernel_regularizer=l2)(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D(2,2)(x)
    x = Dropout(0.25)(x)

    # Block 3
    x = Conv2D(128, (3,3), padding='same', activation='relu', kernel_regularizer=l2)(x)
    x = BatchNormalization()(x)
    x = Conv2D(128, (3,3), padding='same', activation='relu', kernel_regularizer=l2)(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D(2,2)(x)
    x = Dropout(0.25)(x)

    # Block 4
    x = Conv2D(256, (3,3), padding='same', activation='relu', kernel_regularizer=l2)(x)
    x = BatchNormalization()(x)
    x = Conv2D(256, (3,3), padding='same', activation='relu', kernel_regularizer=l2)(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D(2,2)(x)
    x = Dropout(0.25)(x)

    # Head
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation='relu', kernel_regularizer=l2)(x)
    x = Dropout(0.5)(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2)(x)
    x = Dropout(0.4)(x)
    out = Dense(num_classes, activation='softmax')(x)

    model = Model(inp, out, name='cnn_baseline')
    return model

cnn_model = build_cnn()
cnn_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)
cnn_model.summary()

## 1.3 Train CNN

In [ ]:
cnn_callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_cnn.keras', monitor='val_accuracy', save_best_only=True, verbose=0)
]

history_cnn = cnn_model.fit(
    cnn_train_data,
    epochs=EPOCHS,
    validation_data=cnn_val_data,
    callbacks=cnn_callbacks
)

## 1.4 Evaluate CNN

In [ ]:
cnn_pred = cnn_model.predict(cnn_test_data)
cnn_acc  = accuracy_score(cnn_test_data.classes, np.argmax(cnn_pred, axis=1))
print(f'CNN Test Accuracy: {cnn_acc:.4f}')
print(classification_report(
    cnn_test_data.classes,
    np.argmax(cnn_pred, axis=1),
    target_names=EMOTIONS
))

## 1.5 Plot CNN History

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history_cnn.history['accuracy'],     label='Train')
ax1.plot(history_cnn.history['val_accuracy'], label='Val')
ax1.set_title('CNN — Accuracy'); ax1.legend(); ax1.grid(True)

ax2.plot(history_cnn.history['loss'],     label='Train')
ax2.plot(history_cnn.history['val_loss'], label='Val')
ax2.set_title('CNN — Loss'); ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.savefig('cnn_history.png', dpi=150)
plt.show()

---
# Part 2 — MobileNetV2 Transfer Learning
Following the approach from the reference notebook:
- `preprocessing_function = mobilenet_v2.preprocess_input` (handles [-1,1] scaling correctly)
- **All layers trainable** from the start (no freeze/unfreeze stages)
- `ReduceLROnPlateau(factor=0.2, patience=2)` does automatic fine-tuning via LR drops
- `label_smoothing=0.1`

## 2.1 Data Generators for MobileNetV2

In [ ]:
# MobileNetV2 requires its own preprocess_input (scales pixels to [-1, 1])
tl_train_gen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    validation_split=0.2,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

tl_train_data = tl_train_gen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    classes=EMOTIONS
)

tl_val_data = tl_train_gen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    classes=EMOTIONS
)

tl_test_gen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input
)
tl_test_data = tl_test_gen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
    classes=EMOTIONS
)

## 2.2 Build MobileNetV2 Model

In [ ]:
inputs    = Input(shape=(128, 128, 3))
base      = MobileNetV2(weights='imagenet', include_top=False, input_tensor=inputs)

# All layers trainable from the start — LR schedule handles gradual fine-tuning
base.trainable = True

x = base.output

# Additional convolutional head on top of MobileNetV2 features
x = Conv2D(64,  (3,3), activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = Conv2D(128, (3,3), activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = MaxPooling2D(pool_size=(2,2))(x)

x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)

model_tl = Model(inputs, outputs, name='mobilenetv2_fer')

model_tl.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

model_tl.summary()

## 2.3 Train MobileNetV2

In [ ]:
tl_callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_mobilenet.keras', monitor='val_accuracy', save_best_only=True, verbose=0)
]

history_tl = model_tl.fit(
    tl_train_data,
    epochs=EPOCHS,
    validation_data=tl_val_data,
    callbacks=tl_callbacks
)

## 2.4 Evaluate MobileNetV2

In [ ]:
tl_pred = model_tl.predict(tl_test_data)
tl_acc  = accuracy_score(tl_test_data.classes, np.argmax(tl_pred, axis=1))
print(f'MobileNetV2 Test Accuracy: {tl_acc:.4f}')
print(classification_report(
    tl_test_data.classes,
    np.argmax(tl_pred, axis=1),
    target_names=EMOTIONS
))

## 2.5 Plot MobileNetV2 History

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history_tl.history['accuracy'],     label='Train')
ax1.plot(history_tl.history['val_accuracy'], label='Val')
ax1.set_title('MobileNetV2 — Accuracy'); ax1.legend(); ax1.grid(True)

ax2.plot(history_tl.history['loss'],     label='Train')
ax2.plot(history_tl.history['val_loss'], label='Val')
ax2.set_title('MobileNetV2 — Loss'); ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.savefig('tl_history.png', dpi=150)
plt.show()

---
# Final Comparison

In [ ]:
print('=' * 40)
print(f'  CNN Baseline      : {cnn_acc*100:.2f}%')
print(f'  MobileNetV2 (TL)  : {tl_acc*100:.2f}%')
print(f'  Improvement       : {(tl_acc - cnn_acc)*100:+.2f}%')
print('=' * 40)

## Save Models

In [ ]:
cnn_model.save('fer_cnn_model.keras')
model_tl.save('fer_mobilenetv2_model.keras')
print('Both models saved.')